# V2-01 — Topic **cascades** across the journal panel (exploratory cartography)

**Exploratory, not confirmatory.** This notebook renders the V2-S03 cascade engine
outputs on the current 10-journal corpus. It makes **no causal claim** and enters
**no hypothesis gate** — V2-S04 supplies the null/permutation + jackknife validation
that decides whether the structure shown here is real or a panel artifact.

> ## ⚠️ Panel-conditional caveat (read before interpreting any panel below)
> An **"origin journal"** here means **"the first journal *in our ten-journal panel*
> to publish the topic"** — *never* "first in the world". The true origin may be a
> journal we never harvested. Likewise a **lead-lag** edge says journal *i* reached a
> shared topic earlier *within the panel*; it is not evidence of intellectual
> priority. The cascade is **orthogonal to the null F1** (F1 asked whether evidence
> *quality* leads *volume* within a topic — null; this asks *which journal publishes
> topic T first* — a different question).
>
> A second, load-bearing caveat: the corpus **starts in 1995**, so a topic already
> present in several journals at corpus inception is **left-censored** — it
> "originates" in 1995 with many co-earliest journals (a tie). ~80% of leaf topics
> tie at the origin year for exactly this reason; treat 1995 origins as *"present at
> panel start"*, not *"born in 1995"*.

The cascade logic lives in `scifield.cartography.cascade` (pure); this notebook does
only I/O + plotting, reading the tables `V2/scripts/build_cascade.py` already wrote.


## 1. Setup + load the cascade tables

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# Repo-root sniff — this notebook lives at V2/notebooks/, code is under src/.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

CASCADE_DIR = repo_root / "V2" / "data" / "cascade"
FLOW_DIR = repo_root / "V2" / "data" / "flow"
TOPIC_HIERARCHY = repo_root / "data" / "v1" / "topic_hierarchy.parquet"
FIG_DIR = repo_root / "V2" / "notebooks" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DPI = 120

from scifield.cartography.cascade import cascade_edges  # noqa: E402

GRAIN = "leaf"  # render the leaf grain; mid is in the same tables under grain=="mid".

lag = pd.read_parquet(CASCADE_DIR / "lag_matrix.parquet")
origins = pd.read_parquet(CASCADE_DIR / "origin_attribution.parquet")
curves = pd.read_parquet(CASCADE_DIR / "diffusion_curves.parquet")
flow = pd.read_parquet(FLOW_DIR / f"flow_{GRAIN}.parquet")

lag_g = lag[lag["grain"] == GRAIN].copy()
origins_g = origins[origins["grain"] == GRAIN].copy()
curves_g = curves[curves["grain"] == GRAIN].copy()

print(f"grain={GRAIN!r}")
print(f"  lag_matrix:         {len(lag_g)} ordered journal pairs")
print(f"  origin_attribution: {len(origins_g)} topics ({int(origins_g['tie'].sum())} ties)")
print(f"  diffusion_curves:   {len(curves_g)} topics")

# Short, figure-friendly journal labels.
SLUG_LABEL = {
    "spine": "Spine",
    "j_arthroplasty": "J Arthroplasty",
    "clin_orthop_relat_res": "CORR",
    "j_bone_joint_surg_am": "J Bone Joint Surg",
    "arthroscopy": "Arthroscopy",
    "surgery": "Surgery",
    "ann_surg": "Ann Surg",
    "br_j_surg": "Br J Surg",
    "j_am_coll_surg": "J Am Coll Surg",
    "jama_surg": "JAMA Surg",
}

grain='leaf'
  lag_matrix:         90 ordered journal pairs
  origin_attribution: 149 topics (118 ties)
  diffusion_curves:   149 topics


## 2. Panel (a) — inter-journal lead-lag matrix (heatmap)

`mean_lag[i, j] > 0` means journal *i* reached a shared topic **earlier** than
journal *j* on average (i.e. *i* leads *j*). The matrix is antisymmetric. Rows are
ordered by **net lead** (mean of a journal's row) — top rows are the panel's
systematic *leaders*, bottom rows the *followers*. Panel-conditional: this is lead
*within our ten journals*, not intellectual priority.

In [2]:
slugs = sorted(set(lag_g["journal_i"]) | set(lag_g["journal_j"]))
pivot = lag_g.pivot(index="journal_i", columns="journal_j", values="mean_lag").reindex(
    index=slugs, columns=slugs
)
# Diagonal is undefined (a journal vs itself); fill with 0 for display only.
pivot_arr = pivot.to_numpy(dtype="float64", copy=True)
np.fill_diagonal(pivot_arr, 0.0)
pivot = pd.DataFrame(pivot_arr, index=pivot.index, columns=pivot.columns)

# Order journals by net lead (row mean) so leaders sit at the top.
net_lead = pivot.mean(axis=1).sort_values(ascending=False)
order = net_lead.index.tolist()
pivot = pivot.reindex(index=order, columns=order)
labels = [SLUG_LABEL.get(s, s) for s in order]

fig, ax = plt.subplots(figsize=(8.6, 7.2))
vmax = np.nanmax(np.abs(pivot.values))
im = ax.imshow(pivot.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(order)))
ax.set_yticks(range(len(order)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
for i in range(len(order)):
    for j in range(len(order)):
        v = pivot.values[i, j]
        if i != j:
            ax.text(j, i, f"{v:+.1f}", ha="center", va="center", fontsize=6.5, color="#222222")
ax.set_title(
    "(a) Inter-journal lead-lag matrix (leaf topics)\n"
    "cell = mean signed year-lag; row leads column when > 0 (warm). "
    "Panel-conditional.",
    fontsize=9,
)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="mean lag (years), row - column lead")
fig.tight_layout()
fig.savefig(FIG_DIR / "v2_01_lag_matrix.png", dpi=DPI, bbox_inches="tight")
plt.close(fig)

print("=== net lead (mean of row; higher = systematically earlier) ===")
print(net_lead.rename(index=SLUG_LABEL).round(2).to_string())
print("\nsaved figures/v2_01_lag_matrix.png")

=== net lead (mean of row; higher = systematically earlier) ===
journal_i
CORR                 2.40
J Bone Joint Surg    2.33
J Am Coll Surg       0.85
Br J Surg            0.82
JAMA Surg            0.28
Surgery             -0.17
Ann Surg            -1.09
Arthroscopy         -1.39
Spine               -1.44
J Arthroplasty      -2.60

saved figures/v2_01_lag_matrix.png


## 3. Panel (b) — directed cascade / seeding network

`scifield.cartography.cascade.cascade_edges` (a thin wrapper over the tested
`directed_seeding_network`) returns a pure edge list `[src, dst, n_precedes,
n_shared, weight]`; a high-`weight` edge `src → dst` means `src` tends to publish a
shared topic **before** `dst`. We need the per-paper / per-cell frame to compute it,
so we feed the flow table (already deduplicated to cells). Drawn with **matplotlib
only** (networkx is not used for drawing — we lay nodes on a circle by net lead),
thresholded at `weight ≥ 0.60`.

In [3]:
# cascade_edges consumes a frame carrying [topic_id, journal_slug, year]; the flow
# table is exactly that (one row per cell — fine, the primitive recomputes first years).
edges = cascade_edges(flow[["topic_id", "journal_slug", "year"]], topic_key="topic_id")
WEIGHT_THRESHOLD = 0.60
strong = edges[edges["weight"] >= WEIGHT_THRESHOLD].copy()
print(f"network: {len(edges)} directed edges | {len(strong)} with weight >= {WEIGHT_THRESHOLD}")

# Circular layout ordered by net lead (strongest leader at top, clockwise).
node_order = net_lead.index.tolist()
n_nodes = len(node_order)
angles = np.linspace(np.pi / 2, np.pi / 2 - 2 * np.pi, n_nodes, endpoint=False)
pos = {
    slug: (float(np.cos(a)), float(np.sin(a))) for a, slug in zip(angles, node_order, strict=False)
}

# Node colour = specialty (reuse the locked 5-vs-5 split via seeding.specialty_of).
from scifield.findings.seeding import specialty_of  # noqa: E402

spec_color = {"orthopedic": "#4285f4", "general_surgery": "#ea4335", None: "#9aa0a6"}
# Node size ~ how many topics the journal originates.
orig_counts = origins_g["origin_journal_slug"].value_counts().to_dict()

fig, ax = plt.subplots(figsize=(8.4, 8.0))
if len(strong):
    wmin, wmax = strong["weight"].min(), strong["weight"].max()
    for e in strong.itertuples():
        x0, y0 = pos[e.src]
        x1, y1 = pos[e.dst]
        frac = 0.0 if wmax == wmin else (e.weight - wmin) / (wmax - wmin)
        ax.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(
                arrowstyle="-|>",
                color="#555555",
                alpha=0.30 + 0.55 * frac,
                lw=0.8 + 2.2 * frac,
                shrinkA=14,
                shrinkB=14,
                connectionstyle="arc3,rad=0.12",
            ),
        )
for slug in node_order:
    x, y = pos[slug]
    size = 250 + 60 * orig_counts.get(slug, 0)
    ax.scatter([x], [y], s=size, c=spec_color[specialty_of(slug)], edgecolors="white", zorder=3)
    ax.text(x * 1.16, y * 1.16, SLUG_LABEL.get(slug, slug), ha="center", va="center", fontsize=8)
ax.set_xlim(-1.45, 1.45)
ax.set_ylim(-1.45, 1.45)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(
    "(b) Directed cascade / seeding network (leaf topics)\n"
    f"arrow src -> dst: src publishes first (weight >= {WEIGHT_THRESHOLD}); "
    "node size ~ #topics originated; colour = specialty. Panel-conditional.",
    fontsize=9,
)
# Legend.
for sp, c in [("orthopedic", "#4285f4"), ("general_surgery", "#ea4335")]:
    ax.scatter([], [], c=c, label=sp, s=120, edgecolors="white")
ax.legend(loc="lower right", fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "v2_01_cascade_network.png", dpi=DPI, bbox_inches="tight")
plt.close(fig)
print("saved figures/v2_01_cascade_network.png")

network: 90 directed edges | 6 with weight >= 0.6
saved figures/v2_01_cascade_network.png


## 4. Panel (c) — diffusion / adoption curve

Per topic, `diffusion_curve` fits years-since-origin → cumulative fraction of the
ten journals reached (empirical CDF + a logistic `t50`/rate). Here we draw the
**corpus-aggregate** adoption curve (mean cumulative reach across topics, by
year-since-origin) plus the distribution of per-topic half-saturation times
`t50_empirical`. A topic that diffuses fast climbs steeply; one confined to a few
journals plateaus low (`reach_fraction < 1`).

In [4]:
# Rebuild each topic's step CDF from the flow first-years to draw an aggregate curve.
from scifield.cartography.cascade import _first_appearance_frame  # noqa: E402

first = _first_appearance_frame(flow, topic_key="topic_id")
max_offset = 20  # cap the x-axis at 20 years since origin for readability.

# For each topic, cumulative fraction of the 10-journal panel reached by each offset.
agg = np.zeros(max_offset + 1)
n_topics = 0
for _, grp in first.groupby("topic_id"):
    years = np.sort(grp["first_year"].to_numpy())
    offsets = years - years[0]
    reached = np.zeros(max_offset + 1)
    for k, off in enumerate(offsets, start=1):
        off = min(int(off), max_offset)
        reached[off:] = k
    agg += reached / 10.0  # fraction of the full panel
    n_topics += 1
agg /= n_topics

fig, (axc, axh) = plt.subplots(1, 2, figsize=(13.5, 5.2), gridspec_kw={"width_ratios": [1.3, 1]})

axc.plot(range(max_offset + 1), agg, marker="o", ms=4, lw=2, color="#34a853")
axc.axhline(curves_g["reach_fraction"].mean(), ls=":", c="#888888", lw=1)
axc.text(
    max_offset,
    curves_g["reach_fraction"].mean(),
    f" mean ceiling {curves_g['reach_fraction'].mean():.2f}",
    va="bottom",
    ha="right",
    fontsize=8,
    color="#666666",
)
axc.set_xlabel("years since panel origin")
axc.set_ylabel("mean fraction of the 10-journal panel reached")
axc.set_ylim(0, 1.0)
axc.set_title(
    "(c) Corpus-aggregate adoption curve\n"
    f"mean over {n_topics} leaf topics; plateau = mean panel reach. Panel-conditional.",
    fontsize=9,
)

# Distribution of per-topic empirical t50 (years to half the journals reached).
t50 = curves_g.loc[curves_g["n_journals"] > 1, "t50_empirical"].clip(upper=max_offset)
axh.hist(t50, bins=range(0, max_offset + 2), color="#4285f4", edgecolor="white")
axh.axvline(t50.median(), ls="--", c="#ea4335", lw=1.4, label=f"median {t50.median():.1f}y")
axh.set_xlabel("t50 — years to reach half the journals a topic reaches")
axh.set_ylabel("number of topics")
axh.set_title("(c2) Per-topic half-saturation time", fontsize=9)
axh.legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "v2_01_diffusion_curve.png", dpi=DPI, bbox_inches="tight")
plt.close(fig)
print(
    f"aggregate curve over {n_topics} topics | "
    f"mean reach ceiling {curves_g['reach_fraction'].mean():.2f} | "
    f"median t50 {t50.median():.1f}y | median span {curves_g['span_years'].median():.0f}y"
)
print("saved figures/v2_01_diffusion_curve.png")

aggregate curve over 149 topics | mean reach ceiling 0.61 | median t50 0.2y | median span 12y
saved figures/v2_01_diffusion_curve.png


## 5. Panel (d) — origin-attribution table with topic top-words

A handful of topics with their **origin journal**, origin year, how many of the ten
journals they reached, and their BERTopic `top_words`. The `tie` flag marks
co-earliest origins (mostly the 1995 left-censoring described up top). Read each row
as *"within our panel, topic T first surfaced in journal J"* — not a priority claim.

In [5]:
th = pd.read_parquet(TOPIC_HIERARCHY, columns=["topic_id", "top_words"])
th["top_words_str"] = th["top_words"].apply(
    lambda w: ", ".join(list(w)[:5]) if w is not None else ""
)
words = dict(zip(th["topic_id"], th["top_words_str"], strict=False))

tbl = origins_g.copy()
tbl["top_words"] = tbl["topic_id"].map(words)
tbl["origin_journal"] = (
    tbl["origin_journal_slug"].map(SLUG_LABEL).fillna(tbl["origin_journal_slug"])
)

# Show the richest cascades (reach all 10) — the most interpretable rows.
show = (
    tbl.sort_values(["n_journals", "topic_id"], ascending=[False, True])
    .head(10)[["topic_id", "top_words", "origin_journal", "origin_year", "n_journals", "tie"]]
    .reset_index(drop=True)
)
print("=== origin attribution — 10 richest-cascade leaf topics (panel-conditional) ===")
with pd.option_context("display.max_colwidth", 48, "display.width", 140):
    print(show.to_string(index=False))

print("\n=== origin-journal frequency (which journals most often surface a topic first) ===")
freq = tbl["origin_journal"].value_counts()
print(freq.to_string())
print(f"\ntie rate (co-earliest origins, mostly 1995 left-censoring): {tbl['tie'].mean():.1%}")

=== origin attribution — 10 richest-cascade leaf topics (panel-conditional) ===
 topic_id                                                             top_words    origin_journal  origin_year  n_journals   tie
      1.0 infection, pji, periprosthetic, joint infection, periprosthetic joint              CORR         1995          10  True
     10.0                    residents, training, skills, performance, resident          Ann Surg         1995          10  True
     18.0                        hospitals, care, surgical, mortality, hospital          Ann Surg         1995          10  True
     30.0                              spinal, tumor, tumors, spine, metastatic              CORR         1995          10  True
     35.0                        opioid, opioid use, opioids, use, prescription J Bone Joint Surg         1995          10 False
     41.0                         vte, aspirin, venous, prophylaxis, thrombosis       Arthroscopy         1995          10  True
     51.0        

## 6. Leaf-vs-mid robustness peek (supporting only)

A quick look that the **leader/follower ordering** is not a leaf-grain artifact: the
net-lead ranking at the mid grain should track the leaf ranking (V2-S04 formalises
this with the Spearman ρ ≥ 0.5 robustness bar). This is a sanity peek, **not** the
validation.

In [6]:
lag_mid = lag[lag["grain"] == "mid"].copy()
slugs_mid = sorted(set(lag_mid["journal_i"]) | set(lag_mid["journal_j"]))
pivot_mid = lag_mid.pivot(index="journal_i", columns="journal_j", values="mean_lag").reindex(
    index=slugs_mid, columns=slugs_mid
)
pivot_mid_arr = pivot_mid.to_numpy(dtype="float64", copy=True)
np.fill_diagonal(pivot_mid_arr, 0.0)
pivot_mid = pd.DataFrame(pivot_mid_arr, index=pivot_mid.index, columns=pivot_mid.columns)
net_lead_mid = pivot_mid.mean(axis=1)

cmp = pd.DataFrame(
    {
        "net_lead_leaf": net_lead,
        "net_lead_mid": net_lead_mid.reindex(net_lead.index),
    }
)
rho = cmp["net_lead_leaf"].corr(cmp["net_lead_mid"], method="spearman")
print("=== net-lead by journal: leaf vs mid (higher = leads) ===")
print(cmp.rename(index=SLUG_LABEL).round(2).to_string())
print(f"\nSpearman rho(leaf, mid) net-lead ranking = {rho:.3f}")
print("(supporting peek only — V2-S04 owns the formal leaf-vs-mid robustness verdict)")

=== net-lead by journal: leaf vs mid (higher = leads) ===
                   net_lead_leaf  net_lead_mid
journal_i                                     
CORR                        2.40          2.81
J Bone Joint Surg           2.33          2.01
J Am Coll Surg              0.85          0.95
Br J Surg                   0.82          0.61
JAMA Surg                   0.28          0.59
Surgery                    -0.17         -0.36
Ann Surg                   -1.09         -2.21
Arthroscopy                -1.39         -0.66
Spine                      -1.44         -1.28
J Arthroplasty             -2.60         -2.46

Spearman rho(leaf, mid) net-lead ranking = 0.964
(supporting peek only — V2-S04 owns the formal leaf-vs-mid robustness verdict)


## 7. Summary (exploratory)

- **Leaders / followers:** the older, broad orthopedic journals (CORR, J Bone Joint
  Surg) sit at the top of the net-lead matrix; sub-specialty / newer journals
  (J Arthroplasty, Spine) follow. General-surgery journals split, with Ann Surg the
  single most frequent **origin** journal.
- **Origins are heavily left-censored at 1995** (~80% tie at the panel start) — the
  honest reading is *"present at panel start"*, not *"born then"*. V2-S04 must guard
  this in any origin claim.
- **Diffusion:** the median topic takes ~12 years to reach its last journal and a
  small `t50`, but the **mean panel reach ceiling is ~0.6** — most topics never touch
  all ten journals (cross-specialty topics are the exception).
- Everything here is **panel-conditional and exploratory**. The null/permutation +
  jackknife in **V2-S04** decides whether this lead-lag structure is real or an
  artifact of which ten journals we happened to harvest.